<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Clustering Jerárquico Aglomerativo y Dendrogramas
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 10
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/10%20-%20Clustering/02_Clustering_Jerarquico_Aglomerativo_y_Dendrogramas.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, urllib.request
import warnings
warnings.filterwarnings('ignore')

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (8.5, 4.5)
plt.rcParams['font.size'] = 10

def load_dataset(filename, module_folder="10 - Clustering"):
    local_path = os.path.join(os.getcwd(), "data", filename)
    if os.path.exists(local_path):
        return local_path
    
    parent_path = os.path.join(os.getcwd(), "..", module_folder, "data", filename)
    if os.path.exists(parent_path):
        return parent_path

    raw_url = f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/Data%20Science%20programming/{module_folder.replace(' ', '%20')}/data/{filename}"
    os.makedirs("data", exist_ok=True)
    target_path = os.path.join("data", filename)
    if not os.path.exists(target_path):
        urllib.request.urlretrieve(raw_url, target_path)
    return target_path

print("🚀 Entorno configurado exitosamente para el Módulo 10: Clustering.")


---
### 1. Paradigma de Agrupamiento Jerárquico: Aglomerativo (HAC) vs Divisivo 🌳

A diferencia de K-Means (donde debemos fijar $k$ por anticipado), el **Clustering Jerárquico** genera un árbol completo de relaciones anidadas entre las observaciones:
* **Enfoque Aglomerativo (HAC - *Bottom-Up*):** Comienza con $n$ clusters individuales (cada dato es su propio grupo) y en cada paso fusiona recursivamente los dos clusters más cercanos hasta formar un único gran cluster raíz.
* **Enfoque Divisivo (*Top-Down*):** Comienza con todos los datos en un único cluster y los divide recursivamente.

<div align="center">
  <img src="images/hac_linkage_methods.PNG" width="550" alt="HAC Linkage Methods" style="border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); margin: 10px 0;"/>
</div>


---
### 2. Métodos de Enlace (*Linkage Criteria*) 🔗

La forma en que se calcula la distancia entre dos conjuntos de puntos $A$ y $B$ define el comportamiento del agrupamiento:

| Método de Enlace | Definición Matemática | Propiedades y Casos de Uso |
|---|---|---|
| **Single Linkage** | $d(A, B) = \min \{ d(x, y) : x \in A, y \in B \}$ | Une puntos más cercanos. Susceptible al efecto de "encadenamiento" (*chaining*). |
| **Complete Linkage** | $d(A, B) = \max \{ d(x, y) : x \in A, y \in B \}$ | Une puntos más lejanos. Tiende a producir clusters compactos y esféricos. |
| **Average Linkage (UPGMA)** | $d(A, B) = rac{1}{\|A\|\|B\|} \sum_{x \in A}\sum_{y \in B} d(x, y)$ | Promedio de todas las distancias entre pares. Muy robusto a ruido. |
| **Ward's Method** | $\Delta \text{ESS}_{AB} = rac{\|A\|\|B\|}{\|A\|+\|B\|} \|\mu_A - \mu_B\|^2$ | Minimiza el incremento en la varianza total intra-cluster. Es el estándar recomendado. |


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler

# Cargar dataset de clientes
path_data = load_dataset('mall_customers.csv', '10 - Clustering')
df = pd.read_csv(path_data)

# Seleccionar muestra representativa de 40 clientes para dendrograma claro
muestra = df[['Annual_Income_k', 'Spending_Score']].iloc[:40].values
muestra_scaled = StandardScaler().fit_transform(muestra)

# Calcular matriz de enlace con método Ward
Z_ward = linkage(muestra_scaled, method='ward', metric='euclidean')

print("Forma de la matriz de enlace Z (n-1 fusiones x 4 columnas):", Z_ward.shape)
print("Primeras 3 fusiones [Cluster_1, Cluster_2, Distancia, Conteo]:\n", Z_ward[:3].round(3))


---
### 3. Construcción e Interpretación del Dendrograma 📊

El **Dendrograma** es un diagrama en forma de árbol que ilustra el proceso secuencial de fusión:
* El **eje horizontal** representa las observaciones individuales.
* El **eje vertical** representa la **distancia de fusión** (disimilitud) entre los grupos combinados.
* Una **línea horizontal de corte** a una altura determinada define el número de clusters resultantes.


In [ ]:
plt.figure(figsize=(11, 5.5))
plt.title('Dendrograma Jerárquico de Clientes (Método de Ward)', fontsize=14, fontweight='bold')
plt.xlabel('Índice de la Observación / Muestra', fontweight='bold')
plt.ylabel('Distancia Euclidiana de Fusión', fontweight='bold')

# Graficar dendrograma
dendrogram(
    Z_ward,
    leaf_rotation=90,
    leaf_font_size=9,
    color_threshold=3.5  # Umbral de corte de color
)

plt.axhline(y=3.5, color='r', linestyle='--', label='Umbral de Corte sugerido (k=5)')
plt.legend()
plt.tight_layout()
plt.show()


---
### 4. Ajuste con `AgglomerativeClustering` de Scikit-Learn 🤖


In [ ]:
from sklearn.cluster import AgglomerativeClustering

# Ajustar modelo HAC con 5 clusters
X_all = df[['Annual_Income_k', 'Spending_Score']].values
X_all_scaled = StandardScaler().fit_transform(X_all)

hac = AgglomerativeClustering(n_clusters=5, metric='euclidean', linkage='ward')
hac_labels = hac.fit_predict(X_all_scaled)
df['HAC_Cluster'] = hac_labels

plt.figure(figsize=(8.5, 4.5))
colores = ['#0284c7', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6']
for i in range(5):
    plt.scatter(X_all[hac_labels == i, 0], X_all[hac_labels == i, 1], 
                s=55, c=colores[i], label=f'HAC Grupo {i}', alpha=0.8)

plt.title('Clusters Obtenidos con AgglomerativeClustering (Ward)', fontweight='bold')
plt.xlabel('Ingreso Anual (k USD)')
plt.ylabel('Puntuación de Gasto (1-100)')
plt.legend()
plt.tight_layout()
plt.show()


---
##### 🛠️ Práctica 3: Comparación de Criterios de Enlace (Ward vs Complete vs Single)

**Reto:**
1. Ajusta 3 modelos `AgglomerativeClustering(n_clusters=5)` utilizando los enlaces `'ward'`, `'complete'` y `'single'` sobre `X_all_scaled`.
2. Grafica una figura de 1 fila y 3 columnas comparando las segmentaciones resultantes.
3. Observa el fenómeno de encadenamiento (*chaining*) en el enlace `'single'`.


In [ ]:
# =========================================================================
# TU SOLUCIÓN: Práctica 3 - Comparación de Métodos de Enlace
# =========================================================================

# enlaces = ['ward', 'complete', 'single']
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# ...


<details>
<summary><b>💡 Haz clic aquí para ver la Solución Paso a Paso y Explicación</b></summary>
<br>

```python
enlaces = ['ward', 'complete', 'single']
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

for idx, link in enumerate(enlaces):
    modelo_hac = AgglomerativeClustering(n_clusters=5, metric='euclidean', linkage=link)
    pred_labels = modelo_hac.fit_predict(X_all_scaled)
    
    for c_id in range(5):
        axes[idx].scatter(X_all[pred_labels == c_id, 0], X_all[pred_labels == c_id, 1], s=35, alpha=0.75)
    axes[idx].set_title(f'Enlace: {link.capitalize()}', fontweight='bold')
    axes[idx].set_xlabel('Ingreso Anual (k)')
    if idx == 0:
        axes[idx].set_ylabel('Gasto (1-100)')

plt.tight_layout()
plt.show()
```
</details>


---
### 5. Resumen y Conclusiones del Cuaderno 02 📌

1. **Estructura Jerárquica:** El HAC permite examinar la estructura de datos a diferentes niveles de granularidad mediante un dendrograma sin prefijar $k$.
2. **Criterio de Ward:** Minimiza el incremento de varianza total y es el más utilizado para datos continuos por su robustez y equilibrio.
3. **Costo Computacional:** La complejidad temporal de HAC es $\mathcal{O}(n^2 \log n)$ a $\mathcal{O}(n^3)$, por lo que no es escalable para conjuntos de datos con cientos de miles de registros.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos</i>
  </p>
</div>
